<a href="https://colab.research.google.com/github/xshini01/sd-forge-colab/blob/main/forge-Neo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Stable Diffusion Forge Neo Colab**

In [ ]:
# @markdown **Run this and wait for the public url!!!**
# @markdown ---
import os, subprocess, getpass, sys
from IPython.display import clear_output

model_dict = {
    "NoobAI-XL-v1.1": "https://huggingface.co/Laxhar/noobai-XL-1.1/resolve/main/NoobAI-XL-v1.1.safetensors",
    "CyberRealisticXLPlay_V8": "https://huggingface.co/cyberdelia/CyberRealisticXL/resolve/main/CyberRealisticXLPlay_V8.0_FP16.safetensors",
    "Juggernaut-XL-v9": "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"
}
models = "-- pilih model --" # @param ["-- pilih model --","NoobAI-XL-v1.1","Juggernaut-XL-v9","CyberRealisticXLPlay_V8"] {"allow-input":true}
# @markdown **Ngrok Settings**
# @markdown Gunakan ngrok sebagai tunnel pengganti --share?
use_ngrok = False # @param {type:"boolean"}

# ── Helper ──────────────────────────────────────────────────────────────────
def run(label, cmd, **kwargs):
    """Jalankan shell command, print status, tampilkan log hanya jika error."""
    print(f"  ⏳ {label}...", end=" ", flush=True)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print("❌ GAGAL")
        print("─" * 60)
        if result.stdout.strip():
            print("[stdout]\n", result.stdout.strip())
        if result.stderr.strip():
            print("[stderr]\n", result.stderr.strip())
        print("─" * 60)
        sys.exit(result.returncode)
    else:
        print("✅")

def clone_ext(url):
    url = url.strip()
    repo = url.rstrip('/').split('/')[-1].replace('.git', '')
    run(f"Extension: {repo}", f"git clone {url}", cwd="/content/sd-webui-forge-neo/extensions")

# ── Resolve NGROK API key ────────────────────────────────────────────────────
NGROK = None
if use_ngrok:
    try:
        from google.colab import userdata
        NGROK = userdata.get('NGROK')
        if not NGROK:
            raise ValueError("Secret NGROK kosong")
        print("🔑 Ngrok API key diambil dari Colab Secrets ✅")
    except Exception:
        NGROK = getpass.getpass("🔑 Masukkan Ngrok API key: ").strip()
        if NGROK:
            print("🔑 Ngrok API key berhasil disimpan ✅")
        else:
            print("⚠️  Ngrok API key kosong — ngrok tidak akan digunakan")
            NGROK = None

# ── Install Python 3.13 ──────────────────────────────────────────────────────
print("\n📦 Setup Python 3.13")
run("apt-get update",            "apt-get update -y")
run("Install python3.13",        "apt-get install -y python3.13")
run("Remove old python symlink", "rm -f /usr/local/bin/python /usr/local/bin/pip")
run("Set python3.13 as default", "update-alternatives --install /usr/local/bin/python python /usr/bin/python3.13 2")
run("Download get-pip",          "wget -q https://bootstrap.pypa.io/get-pip.py")
run("Install pip",               "python get-pip.py -q")
run("Install uv",                "pip install -qU uv")

# ── Clone Repo ───────────────────────────────────────────────────────────────
print("\n📁 Clone sd-webui-forge-neo")
run("Clone repo", "git clone https://github.com/Haoming02/sd-webui-forge-classic sd-webui-forge-neo --branch neo")

# ── Setup venv ───────────────────────────────────────────────────────────────
print("\n🐍 Setup Virtual Environment")
run("Create venv",            "uv venv venv --python /usr/bin/python3.13 --seed",     cwd="/content/sd-webui-forge-neo")
run("Install httpx",          "uv pip install --python /usr/bin/python3.13 httpx==0.24.1")
run("Install matplotlib-inline", "uv pip install --python /usr/bin/python3.13 matplotlib-inline")
run("Install ipython",        "uv pip install --python /usr/bin/python3.13 ipython")

# ── Extensions ───────────────────────────────────────────────────────────────
print("\n🧩 Install Extensions")
clone_ext("https://github.com/SignalFlagZ/sd-webui-civbrowser")
clone_ext("https://github.com/DominikDoom/a1111-sd-webui-tagcomplete.git")

# ── Download Model ───────────────────────────────────────────────────────────
if models != "-- pilih model --":
    model_url = model_dict.get(models)
    if model_url:
        print(f"\n🖼️  Download Model: {models}")
        run(f"Downloading {models}", f"wget -q --show-progress -P /content/sd-webui-forge-neo/models/Stable-diffusion {model_url}")
    else:
        print(f"⚠️  Model '{models}' tidak ditemukan di daftar")

# ── Launch ───────────────────────────────────────────────────────────────────
clear_output(wait=True)
print("\n🚀 Launching Forge...")
args = "--uv --xformers --cuda-malloc --cuda-stream --pin-shared-memory --enable-insecure-extension-access"
if NGROK:
    args += f" --ngrok {NGROK}"
else:
    args += " --share"

os.chdir("/content/sd-webui-forge-neo")
!python launch.py {args}